# 02 — Build a Cooperative Task Scheduler (No asyncio!)

**Goal:** Build a working task scheduler using ONLY generators. This is the 'aha' moment.

## The idea

```python
# A scheduler is just:
ready_queue = deque([task_A, task_B, task_C])

while ready_queue:
    task = ready_queue.popleft()
    try:
        command = next(task)       # run until yield
        # handle command (reschedule? sleep? spawn?)
    except StopIteration:
        pass                       # task finished
```

That's it. That's an event loop.

## Step 1 — The simplest scheduler (round-robin)

Build a `Scheduler` class with:
- `self.ready = deque()` — queue of generator tasks
- `add_task(gen)` — append to ready queue
- `run()` — while ready queue has items: pop a task, call `next()`, if it didn't raise StopIteration, put it back

Write two generator "tasks" that print steps and `yield` between each step.

Run them and watch them interleave.

In [ ]:
from collections import deque

# Step 1 — Build Scheduler + test tasks


### Question 2.1
Do the tasks take turns? In what order? You just built cooperative multitasking — each task runs until it voluntarily yields.

*Your answer:*


## Step 2 — Add `sleep` support

Tasks should be able to say "wake me up in N seconds" — just like `await asyncio.sleep(N)`.

Extend your scheduler:
- Add `self.sleeping = []` — a min-heap of `(wake_time, seq, task)` tuples
- When `next(task)` returns a `("sleep", seconds)` tuple, add to the heap instead of ready queue
- At the start of each loop iteration, move due timers from sleeping to ready
- If nothing ready but tasks are sleeping, `time.sleep()` until the next one wakes up

Add a `sleep(seconds)` method that returns the `("sleep", seconds)` tuple.

Test with a fast task (no sleep) and a slow task (`yield sched.sleep(1)`).

In [ ]:
import time
import heapq

# Step 2 — Scheduler with sleep support


### Question 2.2
Does `fast_task` run while `slow_task` is sleeping? How does the scheduler know when to wake up sleeping tasks?

**You just built `asyncio.sleep()`!** The scheduler uses a heap of wake-up times — same as the real event loop.

*Your answer:*


## Step 3 — Add `spawn` (create_task equivalent)

Add a `spawn(gen)` method that returns `("spawn", gen)`.

When the scheduler sees this command:
1. Add the new generator to the ready queue
2. Also put the parent task back in the ready queue

Write a `main()` task that spawns 3 workers and then sleeps.

Compare:
```python
# Your scheduler               # asyncio
yield sched.spawn(worker())     asyncio.create_task(worker())
yield sched.sleep(1)            await asyncio.sleep(1)
```

In [ ]:
# Step 3 — Scheduler with spawn support


## Step 4 — Challenge: Add `gather` support

Implement a way for a task to wait until ALL spawned tasks complete (like `asyncio.gather()`).

Hint — you'll need:
1. A `Task` wrapper class that tracks whether a generator is done
2. A `"wait_tasks"` command that suspends the caller until all specified tasks are done
3. When a task finishes (StopIteration), check if anyone is waiting for it

This is hard. Take your time.

In [ ]:
# Step 4 (challenge) — gather support


## Mastery Test

Your scheduler should run 1000 concurrent countdowns on a single thread:

```python
def countdown(name, n):
    while n > 0:
        print(f'{name}: {n}')
        yield sched.sleep(1)
        n -= 1
    print(f'{name}: BLASTOFF!')

for i in range(1000):
    sched.add_task(countdown(f'T{i}', 5))

sched.run()  # should complete in ~5 seconds, not 5000
```

If 1000 tasks complete in ~5 seconds, you've built a working concurrent runtime.

In [ ]:
# Mastery Test — 1000 concurrent countdowns


## Summary

```
Your Scheduler         =    asyncio event loop
Generator + yield      =    async def + await
yield ("sleep", 2)     =    await asyncio.sleep(2)
yield ("spawn", gen)   =    asyncio.create_task(coro)
ready deque            =    event loop's ready queue
sleeping heap          =    event loop's timer heap
```

**asyncio is just your scheduler + I/O polling + nicer syntax.**

**Next notebook:** async/await desugared — proving the equivalence